# Setup

In [ ]:
import datasets
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from google.colab import drive, userdata
from huggingface_hub import login, hf_hub_download

cache_path = "/content/huggingface_cache"
os.makedirs(cache_path, exist_ok=True)
os.environ['HF_HOME'] = cache_path

if userdata.get('HF_TOKEN'):
  login(token=userdata.get('HF_TOKEN'))
  hf_hub_download(repo_id="sookiemonster/asrs-narratives", filename="utils.py", repo_type="dataset",local_dir=".")

raw_dataset = datasets.load_dataset("sookiemonster/asrs-narratives")

utils.py: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/9.74M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/4.09M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/9.38M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10360 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4441 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9868 [00:00<?, ? examples/s]

In [ ]:
train_ds = raw_dataset['train']
valid_ds = raw_dataset['validation']
test_ds = raw_dataset['test']

labels = train_ds.features['label'].names

id_to_label = { idx : label for idx, label in enumerate(labels) }
label_to_id = { label : idx for idx, label in id_to_label.items() }

In [ ]:
from functools import partial

def filter_labels(ds, to_remove:list):
  to_remove_set = set(to_remove)
  return ds.filter(lambda example : id_to_label[example['label']] not in to_remove_set)

filter_ambiguous = partial(filter_labels, to_remove=['ambiguous'])

filtered_train_ds = filter_ambiguous(train_ds)
filtered_valid_ds = filter_ambiguous(valid_ds)

Filter:   0%|          | 0/10360 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4441 [00:00<?, ? examples/s]

In [ ]:
df = filtered_train_ds.to_pandas()
df.label = df.label.map(id_to_label)
df[df.label == 'humanfactors']

,acn,text,label
1,2109432,Narrative 1 - 'Wheelchair battery form and pro...,humanfactors
2,1974739,Narrative 1 - 'The aircraft at some point in t...,humanfactors
9,2232811,Narrative 1 - 'I was simulating a XC flight to...,humanfactors
12,2205943,Narrative 1 - 'During our arrival into ZZZ we ...,humanfactors
13,2195845,Narrative 1 - 'Prior to departure checked for ...,humanfactors
...,...,...,...
9508,2088537,Narrative 1 - 'Upon reaching Location X; we tr...,humanfactors
9509,2237828,Narrative 1 - 'Agent working in the bagroom; b...,humanfactors
9513,1891350,Narrative 1 - 'Aircraft was operating on an IF...,humanfactors
9514,2144248,Narrative 1 - 'We were coming into BZM; winds ...,humanfactors


In [ ]:
from datasets import concatenate_datasets
ds = concatenate_datasets([raw_dataset['train'], raw_dataset['validation'], raw_dataset['test']])
ds.to_pandas()['label'].map(id_to_label).value_counts().to_csv("prop.csv")
ds.to_pandas()['label'].map(id_to_label).value_counts()

,count
label,
humanfactors,8427
aircraft,8272
procedure,2418
ambiguous,1989
weather,823
airport,610
environment-nonweatherrelated,595
chartorpublication,336
airspacestructure,322


In [ ]:
def _validate_groupings(groupings:dict[str, set]):
  mut_excl = [va.isdisjoint(vb) for ka, va in groupings.items() for kb, vb in groupings.items() if ka != kb]
  assert all(mut_excl), f"{mut_excl}"

  all_labels = set([label for val_set in groupings.values() for label in val_set])
  assert all_labels == set(id_to_label.values()), f"Missing: {set(id_to_label.values()) - all_labels}"


def group_labels(ds, groupings:dict[str, set]):
  _validate_groupings(groupings)
  group_names = list(groupings.keys())
  group_names.sort()

  fine_grained_label_to_group = {
      label : group_name for group_name, val_set in groupings.items() for label in val_set
  }

  res = ds.map(lambda ex: {"group" : fine_grained_label_to_group[ id_to_label[ex['label']] ]})
  res = res.filter(lambda ex: ex["group"] != 'DELETE')

  new_features = res.features.copy()
  group_names.remove("DELETE")
  new_features["group"] = ClassLabel(names=group_names)

  res = res.cast(new_features)
  return res

In [ ]:

def get_inv_class_weights(ds, labels, filter_by='group'):
  length = len(ds)
  counts = [
      len(ds.filter(lambda ex : ex[filter_by] == idx)) for idx, _ in enumerate(labels)
  ]

  return [(length - c) / length for c in counts]


# Fine-Tuning Setup

In [ ]:
import torch
import sys

print(f"Python Version: {sys.version_info.major}.{sys.version_info.minor}")
print(f"PyTorch Version: {torch.__version__.split('+')[0]}")
print(f"CUDA Version (Torch): {torch.version.cuda}")
print(f"CXX11 ABI: {torch._C._GLIBCXX_USE_CXX11_ABI}")

Python Version: 3.12
PyTorch Version: 2.10.0
CUDA Version (Torch): 12.8
CXX11 ABI: True


In [ ]:
!pip install "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.6/253.6 MB 10.5 MB/s eta 0:00:00


In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00


In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")
precison = evaluate.load("precision")
recall = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precison.compute(predictions=predictions, references=labels, zero_division=np.nan)["precision"],
        "recall": recall.compute(predictions=predictions, references=labels)["recall"]
    }

In [ ]:
from transformers import Trainer
import torch.nn as nn

class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(torch.device("cuda")))

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss = self.loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# Distinguish (Human Factors & Procedure) from Rest

In [ ]:
from datasets import ClassLabel

groupings = {
    'DELETE' : set(['aircraft']),
    'hf-proc' : set(['humanfactors', 'procedure']),
    'not-hf-proc' : set(id_to_label.values()) - set(['humanfactors', 'procedure', 'aircraft'])
}

grouped_ds_train = group_labels(filtered_train_ds, groupings)
grouped_ds_valid = group_labels(filtered_valid_ds, groupings)

In [ ]:
get_inv_class_weights(grouped_ds_train, grouped_ds_train.features['group'].names)

Filter:   0%|          | 0/6051 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6051 [00:00<?, ? examples/s]

[0.24723186250206577, 0.7527681374979343]

## Fine Tuning

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "sookiemonster/asrs-modernbert-aircraft-vs-rest"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def preprocess(ds:datasets):
    PREFIX = "classification: "
    res = ds.map(lambda examples : {"text" : PREFIX + examples["text"]})
    res = res.map(lambda examples : tokenizer(
        examples["text"],
        padding=True,
        return_tensors="pt"
    ), batched=True)
    res = res.remove_columns(["acn", "label", 'text']).rename_column('group', 'label')
    return res

tokenized_train = preprocess(grouped_ds_train)
tokenized_eval = preprocess(grouped_ds_valid)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/6051 [00:00<?, ? examples/s]

Map:   0%|          | 0/6051 [00:00<?, ? examples/s]

Map:   0%|          | 0/2594 [00:00<?, ? examples/s]

Map:   0%|          | 0/2594 [00:00<?, ? examples/s]

In [ ]:
id_to_group = { i: group for i, group in enumerate(tokenized_train.features['label'].names) }
group_to_id = { group: i for i, group in id_to_group.items()}

In [ ]:
id_to_group

{0: 'hf-proc', 1: 'not-hf-proc'}

In [ ]:
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding
import torch

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    id2label=id_to_group,
    label2id=group_to_id,
    dtype=torch.bfloat16,
    # dtype=torch.float32,
    num_labels=2,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device=device)

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

In [ ]:
display(grouped_ds_train.features['group'])
CLASS_WEIGHTS = torch.tensor([0.25, 0.75], dtype=torch.float)

ClassLabel(names=['hf-proc', 'not-hf-proc'])

## Run Trainer


In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./modernbert-finetuned",
    bf16=True,

    learning_rate=4e-5,
    num_train_epochs=4,
    weight_decay=0.01,

    per_device_train_batch_size=64,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    push_to_hub=True,
    hub_model_id="sookiemonster/asrs-modernbert-l2-hf-proc",

    report_to="none",
    logging_steps=50,
)

trainer = WeightedLossTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=CLASS_WEIGHTS,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall
1,0.551463,0.473412,0.818813,0.623919,0.674455
2,0.472181,0.464945,0.809561,0.595361,0.719626
3,0.419413,0.462128,0.805705,0.582933,0.755452
4,0.402996,0.460850,0.810717,0.596178,0.728972


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=380, training_loss=0.454377051403648, metrics={'train_runtime': 283.1071, 'train_samples_per_second': 85.494, 'train_steps_per_second': 1.342, 'total_flos': 6.855903039757363e+16, 'train_loss': 0.454377051403648, 'epoch': 4.0})

# Distinguish Weather from Rest

In [ ]:
from datasets import ClassLabel

groupings = {
    'DELETE' : set(['aircraft']),
    'weather' : set(['weather']),
    'not-weather' : set(id_to_label.values()) - set(['weather', 'aircraft'])
}

grouped_ds_train = group_labels(filtered_train_ds, groupings)
grouped_ds_valid = group_labels(filtered_valid_ds, groupings)

Map:   0%|          | 0/9525 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9525 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/6051 [00:00<?, ? examples/s]

Map:   0%|          | 0/4083 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4083 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/2594 [00:00<?, ? examples/s]

In [ ]:
get_inv_class_weights(grouped_ds_train, grouped_ds_train.features['group'].names)

Filter:   0%|          | 0/6051 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6051 [00:00<?, ? examples/s]

[0.05718063130061147, 0.9428193686993885]

## Fine Tuning

In [ ]:
display(grouped_ds_train.features['group'])
CLASS_WEIGHTS = torch.tensor([0.06, 0.94], dtype=torch.float)

ClassLabel(names=['not-weather', 'weather'])

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "sookiemonster/asrs-modernbert-aircraft-vs-rest"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def preprocess(ds:datasets):
    PREFIX = "classification: "
    res = ds.map(lambda examples : {"text" : PREFIX + examples["text"]})
    res = res.map(lambda examples : tokenizer(
        examples["text"],
        padding=True,
        return_tensors="pt"
    ), batched=True)
    res = res.remove_columns(["acn", "label", 'text']).rename_column('group', 'label')
    return res

tokenized_train = preprocess(grouped_ds_train)
tokenized_eval = preprocess(grouped_ds_valid)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/6051 [00:00<?, ? examples/s]

Map:   0%|          | 0/6051 [00:00<?, ? examples/s]

Map:   0%|          | 0/2594 [00:00<?, ? examples/s]

Map:   0%|          | 0/2594 [00:00<?, ? examples/s]

In [ ]:
id_to_group = { i: group for i, group in enumerate(tokenized_train.features['label'].names) }
group_to_id = { group: i for i, group in id_to_group.items()}

In [ ]:
id_to_group

{0: 'not-weather', 1: 'weather'}

In [ ]:
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding
import torch

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    id2label=id_to_group,
    label2id=group_to_id,
    dtype=torch.bfloat16,
    # dtype=torch.float32,
    num_labels=2,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device=device)

model.safetensors:   0%|          | 0.00/299M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

## Run Trainer


In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./modernbert-finetuned",
    bf16=True,

    learning_rate=4e-5,
    num_train_epochs=4,
    weight_decay=0.01,

    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    push_to_hub=True,
    hub_model_id="sookiemonster/asrs-modernbert-l2-weather-mini-batch",

    report_to="none",
    logging_steps=50,
)

trainer = WeightedLossTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=CLASS_WEIGHTS,
)

In [ ]:
# Batch size 16
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall
1,0.342996,0.376473,0.972629,0.873786,0.608108
2,0.183791,0.294954,0.975328,0.772727,0.804054
3,0.289716,0.306393,0.975328,0.772727,0.804054
4,0.080340,0.308265,0.975713,0.774194,0.810811


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1516, training_loss=0.27370150375492025, metrics={'train_runtime': 432.0579, 'train_samples_per_second': 56.02, 'train_steps_per_second': 3.509, 'total_flos': 6.822995482200903e+16, 'train_loss': 0.27370150375492025, 'epoch': 4.0})

In [ ]:
# Batch size 128
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall
1,No log,0.183135,0.945644,0.513834,0.878378
2,0.445896,0.172090,0.961449,0.617647,0.851351
3,0.200336,0.185911,0.934850,0.463158,0.891892
4,0.201379,0.174951,0.945258,0.511719,0.885135


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=192, training_loss=0.2574991335471471, metrics={'train_runtime': 291.1555, 'train_samples_per_second': 83.131, 'train_steps_per_second': 0.659, 'total_flos': 6.855903039757363e+16, 'train_loss': 0.2574991335471471, 'epoch': 4.0})